Create a .env file and add the following:

OPENAI_API_KEY="sk-..." [optional since you can also use Ollama]

EXCHANGE_RATE_API_KEY="..."

In [1]:
# from dotenv import load_dotenv

import os

from typing import Type
from crewai.tools import BaseTool
from crewai import Agent, Task, Crew, Process

from pydantic import BaseModel, Field

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
# load_dotenv()

In [2]:
# from crewai import LLM 
# llm = LLM(
#     model="gpt-4o-mini",
# )

In [3]:
from crewai import LLM 

llm = LLM(
    model="ollama/llama3.2:3b",
    base_url="http://localhost:11434"
)

In [ ]:
class DummyDAQAPI:
    def check_connectivity(self, daq_serial_number: str) -> bool:
        # In this example, we return True only if the serial number 
        # is "TN0000192929029".
        return daq_serial_number == "TN0000192929029"

In [ ]:


# Define the Currency Converter Tool
class InstallationManualToolInput(BaseModel):
    """Input schema for InstallationManualTool."""
    question: str = Field(..., description="The question from the user.")

class InstallationManualTool(BaseTool):
    name: str = "Installation Manual Tool"
    description: str = "Returns content of the installation manual that is relevant to the user's question."
    args_schema: Type[BaseModel] = InstallationManualToolInput

    def _run(self, question: str) -> str:
        script_dir = os.getcwd()
        file_path = os.path.join(script_dir, "docs", "scraped_docs.md")
        loader = TextLoader(file_path=file_path, encoding="utf-8")
        data = loader.load()

        # split the text into chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        all_splits = text_splitter.split_documents(data)

        # embed the chunks in a vector space
        local_embeddings = OllamaEmbeddings(model="nomic-embed-text")
        vectorstore = Chroma.from_documents(documents=all_splits, embedding=local_embeddings)

        # check whether similarity search works
        docs = vectorstore.similarity_search(question)

        return "\n\n".join(doc.page_content for doc in docs)




In [5]:
question_analyst = Agent(
    role="Question Analyst",
    goal="Analyze the user's question and output a structured response containing the a concise version of the question. The question is: '{question}'.",
    backstory=(
        "You are a language understanding expert and understand how to extract core information from questions."
        "You understand the user's natural language question and output a concise version of it."
    ),
    verbose=True,
    llm=llm,
)

question_task = Task(
    description="Understand the user's natural language question and extract the core question. The question is: '{question}'.",
    expected_output="A structured and consicse version to the user's question.",
    agent=question_analyst,
    output_pydantic=InstallationManualToolInput,
)

In [6]:
installation_manual_searcher = Agent(
    role="Installation Manual Searcher",
    goal="Find the content in the installation manual...",
    backstory=(
        "You know how to use the installation manual to find the information..."
        "When you need to search, call the tool with Action: Installation Manual Tool..."
        "The tool will return the content of the installation manual that is relevant to the user's question."
    ),
    verbose=True,
    tools=[InstallationManualTool()],  # Make sure it is `tools=` not `tool=`
    llm=llm,
)


installation_manual_search_task = Task(
    description=(
        "Accept the output form the question analyst agent and return the content of the installation"
        "manual that is relevant to the user's question."
    ),
    expected_output=(
        "An concise answer based on output of the installation manual tool."
        "If you don't know the answer, just say that you don't know."
        "Use three sentences maximum and keep the answer concise"
    ),
    agent=installation_manual_searcher,
)

In [7]:

# # Define the Agent
# currency_analyst = Agent(
#     role="Currency Analyst",
#     goal="Provide real-time currency conversions and financial insights.",
#     backstory=(
#         "You are a finance expert with deep knowledge of global exchange rates."
#         "You help users with currency conversion and financial decision-making."
#     ),
#     tools=[CurrencyConverterTool()],
#     verbose=True,
#     llm=llm
# )

# # Define a Task
# currency_conversion_task = Task(
#     description=(
#         "Accept the output from the query analyst agent and convert the amount using real-time exchange rates."
#         "The input will contain the amount, from_currency, and to_currency."
#         "Provide the equivalent amount and explain any relevant financial context."
#     ),
#     expected_output="A detailed response including the converted amount and financial insights.",
#     agent=currency_analyst
# )

In [8]:

# # Form the Crew
# crew = Crew(
#     agents=[query_analyst, currency_analyst],
#     tasks=[query_task, currency_conversion_task],
#     process=Process.sequential
# )

# response = crew.kickoff(inputs={"query": "How much is 100 USD in EUR?"})

# Form the Crew
crew = Crew(
    agents=[question_analyst, installation_manual_searcher],
    tasks=[question_task, installation_manual_search_task],
    process=Process.sequential
)

question = "LED 1 of my SAMOTICS DAQ flashes red, what does this mean?"

response = crew.kickoff(inputs={"question": question})

# Agent: Question Analyst
## Task: Understand the user's natural language question and extract the core question. The question is: 'LED 1 of my SAMOTICS DAQ flashes red, what does this mean?'.


# Agent: Question Analyst
## Final Answer: 
{"question": "LED 1 of my SAMOTICS DAQ flashes red, what does this mean?"}


# Agent: Installation Manual Searcher
## Task: Accept the output form the question analyst agent and return the content of the installationmanual that is relevant to the user's question.


# Agent: Installation Manual Searcher
## Using tool: Installation Manual Tool
## Tool Input: 
"{\"question\": \"LED 1 of my SAMOTICS DAQ flashes red, what does this mean?\"}"
## Tool Output: 
LED 1 flashes blue during a boot session. If it stays in the blue state, the DAQ needs replacement.
If LED 1 flashes red, it means that the DAQ could not find the gateway. For this scenario, please check the cabling from switch to gateway first, followed by the cable between the DAQ and the switch
Noti

In [9]:
from IPython.display import Markdown
Markdown(response.raw)

If LED 1 flashes red, it means that the DAQ could not find the gateway. For this scenario, please check the cabling from switch to gateway first, followed by the cable between the DAQ and the switch